# 02 · Baseline Léxico (BM25)

Este notebook implementa un baseline de recuperación léxica usando BM25.

Objetivos:
- Construir un índice BM25 sobre el catálogo de muestra.
- Ejecutar búsquedas léxicas para las consultas de desarrollo.
- Evaluar el baseline con métricas nDCG@10, Recall@10 y MRR@10.
- Comparar resultados con el sistema vectorial (en notebooks posteriores).
- Identificar limitaciones del enfoque léxico.

Este baseline sirve como punto de referencia para justificar el uso de embeddings.


#### Importar librerías y cargar datos

In [1]:
import sys
import os

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

print("Ruta añadida al PYTHONPATH:", ROOT_DIR)

Ruta añadida al PYTHONPATH: /home/alexd/modulo_vector_bbdd/actividad_evaluable


In [2]:
import pandas as pd
import numpy as np
from rank_bm25 import BM25Okapi

from src.utils import safe_read_csv, log_section
from src.metrics import ndcg_at_k, recall_at_k, mrr_at_k
from src.config import ESCI_MAPPING, TOP_K

#### Cargar catálogo y consultas

In [3]:
log_section("Cargar catálogo y consultas")

df_catalog = safe_read_csv("../data/catalogo_muestra.csv")
df_queries = safe_read_csv("../data/consultas_desarrollo.csv")
df_relevances = safe_read_csv("../data/relevancias_desarrollo.csv")

df_catalog.head()


[AURUM] 
[AURUM] ============================================================
[AURUM] Cargar catálogo y consultas
[AURUM] ============================================================
[AURUM] [CSV] Cargado: ../data/catalogo_muestra.csv (1500 filas)
[AURUM] [CSV] Cargado: ../data/consultas_desarrollo.csv (8 filas)
[AURUM] [CSV] Cargado: ../data/relevancias_desarrollo.csv (248 filas)


,record_id,product_id,title,brand,color,locale,text,catalog_version,active
0,000bd6e8-a995-56d0-ba03-559885ccef39,B0818K237B,Kanlin1986 Vestido Largo De Navidad para Mujer...,KanLin1986-Ropa,Negro,es,Kanlin1986 Vestido Largo De Navidad para Mujer...,1,True
1,0037a9df-8492-508f-8167-c09624801216,B086YX9RK5,IQOS Kit Iqos 3 Duo Blue Opk 1 200 g,IQOS,NaN,es,IQOS Kit Iqos 3 Duo Blue Opk 1 200 g. Marca: I...,1,True
2,003a8544-5ab1-5c8e-8fa5-612989e4a7a8,B07FRXCFJ1,ELINKUME Lámpara de pie regulable LED Lámpara ...,ELINKUME,Lámparas de Pie-espiral Estilo-regulable Led,es,ELINKUME Lámpara de pie regulable LED Lámpara ...,1,True
3,0058d463-2a7c-592c-b73c-ea7462fc566b,B0869L7SSX,Guillermo Lenteja Beluga Caviar Negra Salamanc...,Guillermo,NaN,es,Guillermo Lenteja Beluga Caviar Negra Salamanc...,1,True
4,0095c6f6-248f-52a1-87f3-cf0b537507bc,B0831CX4LK,COSTWAY Mesa de Ordenador Escritorio Esquina e...,COSTWAY,Marrón,es,COSTWAY Mesa de Ordenador Escritorio Esquina e...,1,True


#### Construcción del índice BM25

In [4]:
log_section("Construcción del índice BM25")

# Texto del catálogo
corpus = (df_catalog["title"].astype(str) + " " + df_catalog["text"].astype(str)).tolist()

# Tokenización simple
tokenized_corpus = [doc.lower().split() for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus)

print("Índice BM25 construido con", len(tokenized_corpus), "documentos.")


[AURUM] 
[AURUM] ============================================================
[AURUM] Construcción del índice BM25
[AURUM] ============================================================
Índice BM25 construido con 1500 documentos.


#### Función de búsqueda BM25

In [5]:
def bm25_search(query_text, top_k=TOP_K):
    tokens = query_text.lower().split()
    scores = bm25.get_scores(tokens)

    # Top-k documentos
    top_idx = np.argsort(scores)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_idx, start=1):
        row = df_catalog.iloc[idx]
        results.append({
            "rank": rank,
            "record_id": row["record_id"],
            "product_id": row["product_id"],
            "title": row["title"],
            "brand": row["brand"],
            "color": row["color"],
            "score": scores[idx]
        })

    return results


#### Evaluación del baseline

In [6]:
log_section("Evaluación del baseline BM25")

ndcgs = []
recalls = []
mrrs = []

for _, row in df_queries.iterrows():
    query_id = row["query_id"]
    query_text = row["query_text"]

    # Relevancias de la consulta
    df_rel = df_relevances[df_relevances["query_id"] == query_id]

    # Resultados BM25
    results = bm25_search(query_text, top_k=TOP_K)

    # Construir vector de relevancias
    relevances = []
    for res in results:
        product_id = res["product_id"]
        rel_row = df_rel[df_rel["product_id"] == product_id]

        if len(rel_row) == 0:
            relevances.append(0)
        else:
            esc = rel_row.iloc[0]["esci_label"]
            relevances.append(ESCI_MAPPING[esc])

    # Métricas
    ndcgs.append(ndcg_at_k(relevances, TOP_K))
    recalls.append(recall_at_k(relevances, TOP_K))
    mrrs.append(mrr_at_k(relevances, TOP_K))

bm25_metrics = {
    "ndcg@10": float(np.mean(ndcgs)),
    "recall@10": float(np.mean(recalls)),
    "mrr@10": float(np.mean(mrrs)),
}

bm25_metrics


[AURUM] 
[AURUM] ============================================================
[AURUM] Evaluación del baseline BM25
[AURUM] ============================================================


{'ndcg@10': 0.8826576794389869,
 'recall@10': 0.41350732596955486,
 'mrr@10': 0.9375}

## Resultados del baseline

### Resultados del baseline BM25

Los resultados obtenidos son:

- nDCG@10: 0.8827
- Recall@10: 0.4135
- MRR@10: 0.9375

Interpretación:

- El valor de nDCG@10 es alto, lo que indica que BM25 ordena razonablemente bien los resultados relevantes en las primeras posiciones cuando la consulta coincide léxicamente con el catálogo.
- MRR@10 es muy alto (0.9375), lo que sugiere que en muchas consultas el primer resultado relevante aparece en posiciones muy tempranas.
- Recall@10 es moderado (0.41): BM25 recupera algunos productos relevantes, pero deja fuera una parte importante, especialmente en consultas con sinónimos, variaciones lingüísticas o descripciones largas.
- Aunque BM25 funciona bien en consultas léxicas directas, su rendimiento cae en consultas semánticas, lo que justifica la necesidad de un sistema vectorial basado en embeddings.

Estos resultados confirman que BM25 es un buen baseline, pero insuficiente para capturar relaciones semánticas más complejas presentes en el catálogo.


### Interpretación de métricas

Los resultados del baseline BM25 muestran un comportamiento mixto:

- **nDCG@10 (0.8827)**  
  Indica que BM25 ordena bien los resultados relevantes cuando la consulta coincide léxicamente con el catálogo. La calidad del ranking es buena en escenarios donde el vocabulario del usuario y del producto es similar.

- **MRR@10 (0.9375)**  
  Refleja que, en muchas consultas, el primer resultado relevante aparece en posiciones muy tempranas. BM25 funciona especialmente bien en consultas directas y con términos específicos.

- **Recall@10 (0.4135)**  
  Es la métrica más débil. BM25 recupera menos de la mitad de los productos relevantes dentro del top‑10. Esto confirma que el enfoque léxico pierde recall en consultas con sinónimos, variaciones lingüísticas o descripciones largas.

En conjunto, BM25 es un baseline sólido para coincidencias léxicas, pero insuficiente para capturar relaciones semánticas profundas. Esto justifica la necesidad de un sistema vectorial basado en embeddings.

## Limitaciones del enfoque léxico

### Limitaciones del baseline léxico

BM25 presenta varias limitaciones para este problema, incluso aunque sus métricas iniciales sean razonablemente buenas:

#### 1. No entiende semántica
Aunque BM25 funciona bien cuando la consulta coincide léxicamente con el catálogo, sigue sin comprender relaciones semánticas profundas:

- “soporte monitor” vs “base para pantalla”
- “mesa auxiliar” vs “mesita pequeña”
- “funda tablet” vs “protector para tablet”

#### 2. No maneja sinónimos ni variaciones de lenguaje
El catálogo contiene productos de múltiples categorías (hogar, electrónica, moda), donde aparecen sinónimos y variaciones:

- “negro” vs “Black”
- “blanco” vs “White”
- “chaqueta” vs “cazadora”
- “soporte TV” vs “montura televisor”

BM25 no unifica estas variaciones y pierde recall en consultas con vocabulario diferente al del catálogo.

#### 3. No aprovecha descripciones largas
En el dataset hay descripciones muy extensas (~3000 caracteres).
BM25:

- penaliza textos largos
- no capta relaciones entre frases
- no aprovecha la estructura semántica del texto

Esto afecta especialmente a productos con descripciones técnicas o detalladas.

#### 4. No permite filtros vectoriales
Los filtros por atributos como marca o color:

- deben aplicarse después de la búsqueda
- no pueden integrarse en el scoring
- requieren normalización previa (especialmente color)

Esto limita la flexibilidad del sistema.

#### 5. No escala bien a catálogos grandes
BM25 requiere comparar cada consulta con todos los documentos:

- complejidad lineal
- no aprovecha estructuras ANN
- no es adecuado para catálogos masivos o consultas en tiempo real


### Conclusiones

BM25 proporciona un baseline razonable, con **nDCG@10 y MRR@10 altos**, pero sigue siendo limitado en consultas semánticas.

Las métricas muestran que, aunque BM25 ordena bien los resultados relevantes cuando hay coincidencia léxica, **su Recall@10 es moderado**, lo que indica que deja fuera una parte importante de los productos relevantes.

El sistema vectorial debe mejorar especialmente:

- **Recall@10**, para recuperar más productos relevantes.
- **nDCG@10**, en consultas con sinónimos o variaciones lingüísticas.
- **MRR@10**, en casos donde el primer resultado relevante no coincide léxicamente.
- **latencia**, para búsquedas en catálogos grandes.
- **fidelidad ANN**, para aproximar bien el ranking exacto.

En el siguiente notebook construiremos el sistema de embeddings y compararemos ambos enfoques.

